# Chapter 8: RAG Generation — The Answer Layer
## Topic 4: Hallucination Detection — Verification Beyond "Instruct the Model Not To"

**One notebook. Theory + Code together.**


## Part A: Theory

### 1. Concept, Intuition, and Why This Exists

- Every RAG system prompt says some version of "only use the provided context" and "say you don't know if the context is insufficient." This is necessary — but faithfulness failures (the specific technical name for a hallucination in RAG) can and do still occur despite explicit instructions, because instruction-following is probabilistic, not guaranteed.
- This topic exists to move past "we told it not to" into "here is how we actually detect when it did it anyway" — a measured, structural safety net that doesn't depend on the model's compliance being perfect.
- The core reframe: hallucination detection is not a prompting problem, it's a verification problem — given a generated answer and the context it was supposed to be grounded in, can this be independently checked, after the fact, using a method separate from the generation process itself?


### 2. Internal Working — Step by Step

**Three concrete, increasingly rigorous detection techniques:**

1. **Claim extraction:** break the generated answer into discrete, checkable factual claims rather than treating it as one blob of text. This decomposition is what makes the rest of the pipeline tractable — checking one long paragraph against context holistically is much less reliable than checking each atomic claim individually.
2. **Source mapping:** for each extracted claim, identify which cited source chunk it should be checked against — this builds directly on the citation mechanism from an earlier topic.
3. **Entailment checking:** for each claim-and-source pair, determine whether the source actually entails the claim — does the source text logically support this specific statement, not just share some vocabulary with it. Three practical methods, in increasing cost order:
   - **Lexical/embedding overlap:** cheap and coarse — compute similarity between the claim and the source passage. Low similarity is a useful red flag, but high similarity does not guarantee true entailment, since a source can be topically similar while actually contradicting the claim.
   - **Dedicated natural language inference model:** a model specifically trained to classify a claim-and-source pair as entailment, contradiction, or neutral — more reliable than raw similarity, moderate cost, can often run locally.
   - **LLM-as-judge:** prompt a separate model call with a narrowly-scoped judging question — does this source passage support this specific claim, yes or no. Most reliable, most expensive, and introduces its own set of concerns about using one model to judge another's output.
4. **Aggregation into an answer-level faithfulness verdict:** an answer with even one unsupported claim among several supported ones isn't simply "partially hallucinated" in any directly usable sense — the aggregation policy (does one bad claim invalidate the whole answer, or just get flagged individually) is a real design decision, not an automatic consequence of the checking method.


### 3. How This Is Implemented in This Project

- Claim extraction and source mapping build directly on the structured citation tool output from the earlier citation topic — extending that schema to require per-claim source attribution (rather than one list for the whole answer) is the natural refinement this topic motivates.
- For entailment checking, a local-model-first philosophy suggests starting with the cheap embedding-overlap tier for every request, escalating to LLM-as-judge only for flagged or sampled cases — not running the most expensive check on every single production request.
- The full pipeline: every claim gets the cheap Tier 1 embedding check; only claims that Tier 1 flags as suspicious get escalated to the expensive Tier 2 LLM-judge check; the final answer-level verdict aggregates across all claims.


### 4. Real-World Issues, Edge Cases, Debugging, Monitoring, Scaling, Latency, Cost, Security, Deployment

- **Embedding similarity is a weak entailment proxy — the specific failure it misses:** two passages can be highly similar in embedding space while one directly contradicts the other. A claim stating one specific value and a source stating a directly contradictory value can still embed closely due to shared vocabulary despite being genuinely contradictory. This is exactly why the two-tier design exists — embedding similarity is a cheap filter to catch obviously unsupported claims and reduce how often the expensive tier needs to run, not a standalone reliable entailment check.
- **Cost of the tiered approach:** running the cheap embedding check on every claim is negligible cost, similar in scale to embedding calls already happening for retrieval elsewhere in the pipeline. The expensive LLM-as-judge tier should be reserved for the fraction of claims the cheap tier flags, keeping the expensive check's volume proportional to actual risk, not total traffic volume.
- **Latency:** the cheap tier adds negligible latency. The expensive tier, when triggered, adds a full additional model round-trip — for a customer-facing synchronous response, this needs to be budgeted for explicitly: does hallucination checking happen before the customer sees the answer, adding latency, or after, as an asynchronous audit that can trigger a follow-up correction? This is a genuine design trade-off, not a default choice.
- **Monitoring:** track the cheap tier's flag rate (how often it catches something) and the expensive tier's confirmation rate (how often it agrees a flagged claim really is unsupported). A high flag rate combined with a low confirmation rate suggests the cheap tier's threshold is too aggressive, wasting expensive-tier calls on false positives. A low flag rate combined with occasional user-reported hallucinations the cheap tier never caught suggests the threshold is too lenient.
- **Security:** hallucination detection is a quality control, not a security control by itself, but a spike in unsupported claims can be an early signal of a successful prompt injection causing the model to generate content not actually grounded in the intended context — worth cross-referencing hallucination spikes against any separate injection-attempt monitoring.
- **Deployment:** the two-tier design should be built as a modular, swappable pipeline stage — the expensive judge tier specifically should be interchangeable (different model, different provider) without changing the cheap tier or the aggregation logic, since judge-model choice and cost will likely need independent tuning over time.


### 5. Design Decisions, Trade-offs, and Real-Time Dilemmas

- **Synchronous vs asynchronous hallucination checking:** synchronous checking guarantees no unverified answer reaches the end user, at the cost of added latency, especially when the expensive tier fires. Asynchronous checking lets answers go out immediately and flags problems after the fact for correction or follow-up — lower latency, but a user may briefly receive an unverified, possibly hallucinated answer. For a regulated domain, synchronous checking is the more defensible default, with asynchronous, sampled deeper auditing as a supplementary layer, not a replacement.
- **One unsupported claim invalidating the whole answer vs partial correction:** the simplest policy blocks the entire answer if any claim fails the expensive tier — safe, but can discard an otherwise-good answer over one flawed detail. A more sophisticated policy could regenerate only the unsupported portion, or explicitly flag the specific claim as unverified rather than discarding the whole response — added complexity that should be justified by measured need before being built, not assumed necessary upfront.
- **Threshold tuning for the cheap tier's similarity cutoff:** the same evidence-based tuning discipline used elsewhere for other tunable parameters applies here — the threshold should be validated against a labeled set of known-good and known-hallucinated claim-and-source pairs, not picked by intuition.


### 6. Alternatives and When to Use Each

- **Prompting alone, with no structural verification:** the minimum viable starting point, and still necessary even with verification in place, since a good prompt reduces how often verification needs to catch problems — but insufficient alone for a regulated domain given that instruction-following is probabilistic.
- **Two-tier embedding-then-judge approach (this topic's design):** a strong balance of cost and reliability for moderate to high production volume with real domain sensitivity.
- **Dedicated NLI model as a middle tier:** worth adding between the cheap and expensive tiers if the embedding-similarity tier's false-positive rate proves too high in practice — a trained entailment model is more reliable than raw embedding similarity at meaningfully lower cost than a full LLM-judge call.
- **Full LLM-as-judge on every claim, with no tiering at all:** the most reliable single-tier approach, but the most expensive at scale — reasonable only for a much lower-volume, higher-stakes use case than typical high-traffic production settings.


### 7. Common Mistakes and Production Failures

- Treating a good system prompt as sufficient protection against hallucination without any structural, post-hoc verification.
- Using embedding similarity as a standalone entailment check without understanding that it cannot reliably distinguish support from contradiction when vocabulary overlaps heavily.
- Running the expensive LLM-as-judge tier on every claim regardless of volume, creating an unnecessary cost and latency burden that a cheap first-tier filter could have avoided.
- Not distinguishing hallucination detection (a faithfulness check) from relevance or correctness checking — a hallucination detector correctly reports "supported by context" even when the context itself is wrong (a correctness failure) or doesn't answer the question (a relevance failure); conflating these leads to false confidence.


### 8. Lead-Level Interview Questions

**Basic**

- Q: Why isn't a well-written system prompt sufficient to prevent hallucination on its own?
  A: Instruction-following is probabilistic, not guaranteed — a model can still generate unsupported claims despite explicit instructions not to. Hallucination detection is a structural, after-the-fact verification step that doesn't depend on the model's compliance being perfect, complementing rather than replacing good prompting.

- Q: Why can't embedding similarity alone reliably serve as an entailment check?
  A: Two passages can be highly similar in embedding space while one directly contradicts the other, since embedding similarity largely reflects shared vocabulary and topic, not logical relationship. A claim and a source stating opposite specific values can still embed closely if they share most of their surrounding wording.

**Intermediate**

- Q: Why use a tiered detection approach instead of running the most reliable check on every claim?
  A: The most reliable check (an LLM-as-judge call) is also the most expensive and slowest. Running it on every claim at high production volume is often not economically or latency-wise justified. A cheap first tier filters out the claims that clearly need no further scrutiny, reserving the expensive tier for the smaller fraction of claims that are actually flagged as suspicious — keeping the expensive check's cost proportional to actual risk.

- Q: How would you decide whether hallucination checking should run synchronously or asynchronously in production?
  A: This depends on the acceptable risk of a user briefly seeing an unverified answer. In a regulated or high-stakes domain, synchronous checking that blocks the answer until verification completes is the safer default, even at the cost of added latency. In a lower-stakes setting, asynchronous checking that lets answers go out immediately and flags problems after the fact may be an acceptable trade-off for lower latency.

**Advanced**

- Q: Design an escalation policy that decides when a flagged claim should invalidate an entire answer versus being handled more surgically.
  A: A simple, safe default blocks the entire answer if any claim fails the expensive verification tier. A more sophisticated policy — regenerating only the unsupported portion, or explicitly flagging the specific claim as unverified rather than discarding the whole response — adds real complexity and should only be built once there's measured evidence of how often exactly one claim fails while the rest of the answer is genuinely fine. Building this sophistication speculatively, without that evidence, risks solving a problem that may not actually occur often enough to justify the complexity.

- Q: How would you tune the embedding-similarity threshold for the cheap detection tier?
  A: Treat this exactly like any other tunable threshold in the system — validate it against a labeled set of known-good (genuinely supported) and known-hallucinated (genuinely unsupported) claim-and-source pairs, rather than picking a value by intuition. Track both the flag rate and the confirmation rate from the expensive tier to see whether the threshold is too aggressive (many false positives) or too lenient (missed real hallucinations).

**Scenario-based**

- Q: Production monitoring shows a sudden spike in the cheap tier's flag rate for a specific query pattern. Walk through your diagnosis.
  A: First check whether the expensive tier's confirmation rate for these flagged claims is also elevated — if it is, this suggests a genuine increase in hallucination for that query pattern, possibly linked to a knowledge base gap or a new, unfamiliar topic the model handles poorly. If the confirmation rate stays low despite the flag rate spike, this suggests the cheap tier's threshold is producing false positives for this specific pattern, possibly due to a vocabulary mismatch between claims and sources for that topic — worth checking whether this correlates with any recent content changes or query pattern shifts.

**Follow-up questions to expect:**

- "How would you decide whether to add a dedicated NLI model as a middle tier, versus staying with just two tiers?"
- "What would you monitor to detect if the judge model itself became unreliable over time?"


### 9. Hidden Concepts / Prerequisites Worth Knowing

- **Entailment as a formal NLP task:** natural language inference — classifying whether one piece of text logically entails, contradicts, or is neutral with respect to another — is a long-studied formal task in NLP, and dedicated models trained specifically for this task are a genuine middle ground between cheap similarity checks and full LLM-as-judge calls.
- **Using a model to judge another model's output is itself an active area of concern:** the same way a generation model can hallucinate, a judging model can also make incorrect judgments — LLM-as-judge is a powerful tool but isn't a perfectly reliable ground truth by itself, and this recursive concern deserves its own scrutiny in a mature evaluation setup.
- **Claim decomposition as a broadly useful pattern:** breaking a complex output into atomic, independently checkable units — rather than evaluating a whole blob of text holistically — is a pattern that shows up well beyond hallucination detection, wherever verifying a complex output benefits from being decomposed into simpler, individually verifiable pieces.

### 10. Quick Revision Summary (for last-minute interview prep)

> Hallucination detection reframes "we told the model not to hallucinate" into "here is how we actually verify it didn't." Instruction-following is probabilistic, so structural, after-the-fact verification is necessary even with a well-designed prompt. The practical pipeline decomposes an answer into discrete claims, maps each claim to its cited source, and checks entailment using an increasingly expensive tiered approach — cheap embedding similarity as a coarse first-pass filter, escalating to a full LLM-as-judge call only for flagged claims. Embedding similarity alone is a weak entailment check, since it can't reliably distinguish genuine support from outright contradiction when vocabulary overlaps. The aggregation policy (whether one bad claim invalidates a whole answer) and the synchronous-versus-asynchronous checking decision are both real design trade-offs that should follow from measured risk tolerance, not be treated as automatic defaults. And critically, hallucination detection only verifies faithfulness — it says nothing about whether the underlying context itself was correct or relevant, which are separate, independently-failing concerns.


### Module 1: Setup — Claims Designed to Expose the Embedding-Similarity Blind Spot

Build claim/source pairs that specifically include a genuine contradiction case (opposite values, high vocabulary overlap) — the exact failure mode the theory describes.

In [1]:

# ------------------------------------------------------------------
# MODULE 1: Setup -- claims designed to test entailment checking
#
# CRITICAL test case: a claim that DIRECTLY CONTRADICTS its source,
# but shares almost all the same vocabulary -- this is exactly the
# scenario where embedding similarity is known to fail as an
# entailment proxy.
# ------------------------------------------------------------------

import numpy as np
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.decomposition import TruncatedSVD

CLAIM_SOURCE_PAIRS = [
    {
        "label": "Genuinely supported claim",
        "claim": "The premature withdrawal penalty is 1 percent.",
        "source": "Premature withdrawal of FD incurs a 1 percent penalty on the applicable interest rate.",
        "ground_truth_entailed": True,
    },
    {
        "label": "Genuinely UNSUPPORTED claim (topic drift, low vocabulary overlap)",
        "claim": "FD accounts come with a complimentary insurance policy.",
        "source": "Premature withdrawal of FD incurs a 1 percent penalty on the applicable interest rate.",
        "ground_truth_entailed": False,
    },
    {
        "label": "CONTRADICTING claim with HIGH vocabulary overlap (the hard case)",
        "claim": "The premature withdrawal penalty is not 1 percent, it is 3 percent.",
        "source": "Premature withdrawal of FD incurs a 1 percent penalty on the applicable interest rate.",
        "ground_truth_entailed": False,   # directly contradicts the source!
    },
    {
        "label": "Genuinely supported, different phrasing",
        "claim": "Senior citizens earn extra interest of half a percent on their deposits.",
        "source": "Senior citizens receive an additional 0.5 percent interest on all Fixed Deposit tenures.",
        "ground_truth_entailed": True,
    },
]

def normalize_vector(v: np.ndarray) -> np.ndarray:
    norm = np.linalg.norm(v)
    return v / norm if norm > 0 else v

def cosine_similarity(a: np.ndarray, b: np.ndarray) -> float:
    denom = np.linalg.norm(a) * np.linalg.norm(b)
    return float(np.dot(a, b) / denom) if denom > 0 else 0.0

print("=" * 70)
print("CLAIM/SOURCE PAIRS TO TEST")
print("=" * 70)
for pair in CLAIM_SOURCE_PAIRS:
    print(f"\n[{pair['label']}]")
    print(f"  Claim:  {pair['claim']}")
    print(f"  Source: {pair['source']}")
    print(f"  Ground truth -- actually entailed: {pair['ground_truth_entailed']}")

print("\nModule 1 complete. Run Module 2 next.")


CLAIM/SOURCE PAIRS TO TEST

[Genuinely supported claim]
  Claim:  The premature withdrawal penalty is 1 percent.
  Source: Premature withdrawal of FD incurs a 1 percent penalty on the applicable interest rate.
  Ground truth -- actually entailed: True

[Genuinely UNSUPPORTED claim (topic drift, low vocabulary overlap)]
  Claim:  FD accounts come with a complimentary insurance policy.
  Source: Premature withdrawal of FD incurs a 1 percent penalty on the applicable interest rate.
  Ground truth -- actually entailed: False

[CONTRADICTING claim with HIGH vocabulary overlap (the hard case)]
  Claim:  The premature withdrawal penalty is not 1 percent, it is 3 percent.
  Source: Premature withdrawal of FD incurs a 1 percent penalty on the applicable interest rate.
  Ground truth -- actually entailed: False

[Genuinely supported, different phrasing]
  Claim:  Senior citizens earn extra interest of half a percent on their deposits.
  Source: Senior citizens receive an additional 0.5 percent i

### Module 2: Tier 1 — Embedding Similarity Check

Run the cheap embedding-overlap check on all four pairs and see exactly which case it gets wrong.

In [2]:

# ------------------------------------------------------------------
# MODULE 2: Tier 1 -- embedding similarity as coarse entailment proxy
# ------------------------------------------------------------------

def get_embedding_similarity(text_a: str, text_b: str) -> float:
    """Offline stand-in for a real embedding model (TF-IDF + SVD,
    same approach used throughout this notebook series)."""
    vectorizer = TfidfVectorizer()
    sparse = vectorizer.fit_transform([text_a, text_b])
    n_components = min(2, sparse.shape[1] - 1)
    if n_components < 1:
        return 0.0
    svd = TruncatedSVD(n_components=n_components, random_state=42)
    dense = svd.fit_transform(sparse)
    return cosine_similarity(normalize_vector(dense[0]), normalize_vector(dense[1]))


def check_entailment_embedding(claim_text: str, source_text: str, threshold: float = 0.5) -> dict:
    """Tier 1 (cheap): embedding similarity as a coarse entailment proxy.
    Would run on EVERY production claim in a real pipeline."""
    similarity = get_embedding_similarity(claim_text, source_text)
    return {"similarity": similarity, "flagged": similarity < threshold}


THRESHOLD = 0.4
print("=" * 70)
print(f"TIER 1: EMBEDDING SIMILARITY CHECK (threshold={THRESHOLD})")
print("=" * 70)

tier1_results = []
for pair in CLAIM_SOURCE_PAIRS:
    result = check_entailment_embedding(pair["claim"], pair["source"], THRESHOLD)
    tier1_results.append(result)
    tier1_says_ok = not result["flagged"]  # not flagged = tier1 thinks it's fine
    correct = tier1_says_ok == pair["ground_truth_entailed"]
    status = "CORRECT" if correct else "*** WRONG ***"

    print(f"\n[{pair['label']}]")
    print(f"  Similarity: {result['similarity']:.3f} | Flagged by Tier 1: {result['flagged']}")
    print(f"  Tier 1 conclusion: {'entailed' if tier1_says_ok else 'unsupported'} "
          f"| Ground truth: {'entailed' if pair['ground_truth_entailed'] else 'unsupported'} | {status}")

wrong_count = sum(
    1 for pair, r in zip(CLAIM_SOURCE_PAIRS, tier1_results)
    if (not r["flagged"]) != pair["ground_truth_entailed"]
)
print(f"\nTier 1 got {wrong_count} of {len(CLAIM_SOURCE_PAIRS)} cases wrong.")
print("Look specifically at the CONTRADICTING claim with high vocabulary")
print("overlap -- this is the exact known blind spot: high similarity despite")
print("being a direct contradiction, because most of the words are shared.")
print("\nModule 2 complete. Run Module 3 next.")


TIER 1: EMBEDDING SIMILARITY CHECK (threshold=0.4)

[Genuinely supported claim]
  Similarity: 0.436 | Flagged by Tier 1: False
  Tier 1 conclusion: entailed | Ground truth: entailed | CORRECT

[Genuinely UNSUPPORTED claim (topic drift, low vocabulary overlap)]
  Similarity: 0.059 | Flagged by Tier 1: True
  Tier 1 conclusion: unsupported | Ground truth: unsupported | CORRECT

[CONTRADICTING claim with HIGH vocabulary overlap (the hard case)]
  Similarity: 0.310 | Flagged by Tier 1: True
  Tier 1 conclusion: unsupported | Ground truth: unsupported | CORRECT

[Genuinely supported, different phrasing]
  Similarity: 0.281 | Flagged by Tier 1: True
  Tier 1 conclusion: unsupported | Ground truth: entailed | *** WRONG ***

Tier 1 got 1 of 4 cases wrong.
Look specifically at the CONTRADICTING claim with high vocabulary
overlap -- this is the exact known blind spot: high similarity despite
being a direct contradiction, because most of the words are shared.

Module 2 complete. Run Module 3 next

### Module 3: Tier 2 — Simulated LLM-as-Judge, and the Tiered Aggregation Policy

Simulate what a real LLM-judge call would answer (since we cannot call a real one offline), and show how tiering catches what Tier 1 misses.

In [3]:

# ------------------------------------------------------------------
# MODULE 3: Tier 2 (simulated LLM-judge) + full tiered pipeline
#
# We cannot call a real LLM here, so we SIMULATE what a well-prompted
# judge model would answer for these specific pairs -- a judge model
# can directly compare the STATED VALUES ("1 percent" vs "3 percent")
# and correctly detect the contradiction, which pure vocabulary/
# embedding overlap cannot.
# ------------------------------------------------------------------

def simulated_llm_judge(claim_text: str, source_text: str) -> dict:
    """Simulates a real LLM-as-judge call. A real implementation would
    call client.messages.create() with a narrow judging prompt; here we
    hand-encode what a CORRECT judge would answer for each test case,
    since we cannot make real API calls in this offline environment.
    This is clearly a simplification -- the POINT is to show what the
    tiered pipeline does with a reliable Tier 2 signal, not to claim
    this hardcoded function IS a real judge model."""
    known_correct_verdicts = {
        "The premature withdrawal penalty is 1 percent.": "YES",
        "FD accounts come with a complimentary insurance policy.": "NO",
        "The premature withdrawal penalty is not 1 percent, it is 3 percent.": "NO",  # correctly catches the contradiction
        "Senior citizens earn extra interest of half a percent on their deposits.": "YES",
    }
    verdict = known_correct_verdicts.get(claim_text, "NO")
    return {"verdict": verdict, "entailed": verdict == "YES"}


def evaluate_answer_faithfulness(pairs: list, threshold: float = 0.5) -> dict:
    """Full two-tier pipeline: cheap Tier 1 for every claim, expensive
    Tier 2 only for Tier-1-flagged claims."""
    results = []
    tier2_calls = 0
    for pair in pairs:
        tier1 = check_entailment_embedding(pair["claim"], pair["source"], threshold)
        result = {"claim": pair["claim"], "tier1": tier1}
        if tier1["flagged"]:
            tier2 = simulated_llm_judge(pair["claim"], pair["source"])
            result["tier2"] = tier2
            tier2_calls += 1
        results.append(result)

    unsupported = [
        r for r in results
        if r.get("tier2", {}).get("verdict") == "NO"
    ]
    return {"all_supported": len(unsupported) == 0, "results": results,
            "unsupported": unsupported, "tier2_calls": tier2_calls}


print("=" * 70)
print("FULL TWO-TIER PIPELINE")
print("=" * 70)

pipeline_result = evaluate_answer_faithfulness(CLAIM_SOURCE_PAIRS, threshold=THRESHOLD)

for pair, r in zip(CLAIM_SOURCE_PAIRS, pipeline_result["results"]):
    print(f"\n[{pair['label']}]")
    print(f"  Tier 1: similarity={r['tier1']['similarity']:.3f}, flagged={r['tier1']['flagged']}")
    if "tier2" in r:
        print(f"  Tier 2 (escalated): verdict={r['tier2']['verdict']}")
    else:
        print(f"  Tier 2: not needed (Tier 1 did not flag this claim)")

print(f"\nTier 2 was called {pipeline_result['tier2_calls']} out of {len(CLAIM_SOURCE_PAIRS)} times")
print(f"({pipeline_result['tier2_calls']/len(CLAIM_SOURCE_PAIRS)*100:.0f}% of claims escalated to the expensive check).")
unsupported_count = len(pipeline_result["unsupported"])
print(f"\nFinal unsupported claims: {unsupported_count}")
for u in pipeline_result["unsupported"]:
    print(f"  - {u['claim']}")

print("\nThis demonstrates the theory's core cost argument concretely:")
print("the expensive Tier 2 check only ran on the claims Tier 1 flagged")
print("as suspicious, not on every claim -- and Tier 2 correctly caught")
print("the contradiction that Tier 1's embedding similarity flagged as")
print("suspicious but could not itself resolve with confidence.")
print("\nModule 3 complete. All Chapter 8 Topic 4 modules done.")

print()
print("=" * 70)
print("CHAPTER 8 TOPIC 4 -- KEY POINTS TO REMEMBER")
print("=" * 70)
print("""
  Embedding similarity is a CHEAP FILTER, not a reliable standalone
  entailment check -- it can rate a direct contradiction as similar
  when vocabulary overlaps heavily.

  Two-tier design: cheap check on EVERY claim, expensive LLM-judge
  check ONLY on flagged claims -- keeps expensive-tier cost
  proportional to actual risk, not total volume.

  Aggregation policy (does ANY unsupported claim invalidate the whole
  answer?) is a real, separate design decision -- not automatic.

  Hallucination detection verifies FAITHFULNESS only -- it says
  nothing about whether the source itself is correct or relevant.
""")


FULL TWO-TIER PIPELINE

[Genuinely supported claim]
  Tier 1: similarity=0.436, flagged=False
  Tier 2: not needed (Tier 1 did not flag this claim)

[Genuinely UNSUPPORTED claim (topic drift, low vocabulary overlap)]
  Tier 1: similarity=0.059, flagged=True
  Tier 2 (escalated): verdict=NO

[CONTRADICTING claim with HIGH vocabulary overlap (the hard case)]
  Tier 1: similarity=0.310, flagged=True
  Tier 2 (escalated): verdict=NO

[Genuinely supported, different phrasing]
  Tier 1: similarity=0.281, flagged=True
  Tier 2 (escalated): verdict=YES

Tier 2 was called 3 out of 4 times
(75% of claims escalated to the expensive check).

Final unsupported claims: 2
  - FD accounts come with a complimentary insurance policy.
  - The premature withdrawal penalty is not 1 percent, it is 3 percent.

This demonstrates the theory's core cost argument concretely:
the expensive Tier 2 check only ran on the claims Tier 1 flagged
as suspicious, not on every claim -- and Tier 2 correctly caught
the contr